In [ ]:
import seaborn as sns
import pandas as pd

df = sns.load_dataset('titanic')
df.head()

In [ ]:
df = df.drop(['deck', 'embark_town', 'alive', 'class', 'who', 'adult_male'], axis=1)

X = df.drop('survived', axis=1)
y = df['survived']

X.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train = X_train.copy()
X_test = X_test.copy()

age_median = X_train['age'].median()
embarked_mode = X_train['embarked'].mode()[0]

X_train['age'] = X_train['age'].fillna(age_median)
X_test['age'] = X_test['age'].fillna(age_median)

X_train['embarked'] = X_train['embarked'].fillna(embarked_mode)
X_test['embarked'] = X_test['embarked'].fillna(embarked_mode)

X_train['sex'] = X_train['sex'].map({'male': 0, 'female': 1})
X_test['sex'] = X_test['sex'].map({'male': 0, 'female': 1})

X_train['embarked'] = X_train['embarked'].map({'S': 0, 'C': 1, 'Q': 2})
X_test['embarked'] = X_test['embarked'].map({'S': 0, 'C': 1, 'Q': 2})

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

baseline_model = DecisionTreeClassifier(random_state=42)
cv_scores = cross_val_score(baseline_model, X_train, y_train, cv=5)

baseline_model.fit(X_train, y_train)
baseline_test_acc = accuracy_score(y_test, baseline_model.predict(X_test))

print("Baseline CV Scores:", cv_scores)
print("Baseline CV Accuracy:", cv_scores.mean())
print("Baseline Test Accuracy:", baseline_test_acc)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 5, 10]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)
grid_search.fit(X_train, y_train)

grid_test_acc = accuracy_score(y_test, grid_search.best_estimator_.predict(X_test))

print("Best Parameters from GridSearchCV:", grid_search.best_params_)
print("Best CV Score from GridSearchCV:", grid_search.best_score_)
print("GridSearchCV Test Accuracy:", grid_test_acc)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 5, 10, 15]
}

random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_distributions=param_dist, n_iter=5, cv=5, random_state=42)
random_search.fit(X_train, y_train)

random_test_acc = accuracy_score(y_test, random_search.best_estimator_.predict(X_test))

print("Best Parameters from RandomizedSearchCV:", random_search.best_params_)
print("Best CV Score from RandomizedSearchCV:", random_search.best_score_)
print("RandomizedSearchCV Test Accuracy:", random_test_acc)

In [ ]:
from sklearn.ensemble import BaggingClassifier

bagging_model = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=50, random_state=42)
bag_cv = cross_val_score(bagging_model, X_train, y_train, cv=5).mean()

bagging_model.fit(X_train, y_train)
bag_test_acc = accuracy_score(y_test, bagging_model.predict(X_test))

print("Bagging CV Accuracy:", bag_cv)
print("Bagging Test Accuracy:", bag_test_acc)

In [ ]:
results = pd.DataFrame({
    'Model': ['Decision Tree Baseline', 'GridSearchCV', 'RandomizedSearchCV', 'Bagging'],
    'CV Accuracy': [cv_scores.mean(), grid_search.best_score_, random_search.best_score_, bag_cv],
    'Test Accuracy': [baseline_test_acc, grid_test_acc, random_test_acc, bag_test_acc],
    'Best Params': ['Default', grid_search.best_params_, random_search.best_params_, 'Default']
})

results